In [ ]:
import pandas as pd
import numpy as np
import json

#  Calendar year to Financial year map
fy_map = lambda q: q.year - 1 if q.quarter == 1 else q.year
p = json.load(open("../../data/processed/bridge_params.json"))
seasonal = {1: 0, 2: p["q2"], 3: p["q3"], 4: p["q4"]}

# Seed with last 4 actual log starts from England master
eng = pd.read_csv("../../data/python_master/england_master.csv")
seed = np.log(eng["starts"].dropna().iloc[-4:].values)


def parse_quarter(x):
    x = str(x)
    qmap = {"Jan - Mar": 1, "Apr - Jun": 2, "Jul - Sep": 3, "Oct - Dec": 4}
    for k, v in qmap.items():
        if x.startswith(k):
            try: return pd.Period(year=int(x[-4:]), quarter=v, freq="Q")
            except: return pd.NaT
    return pd.NaT


ons = pd.read_excel("../../data/raw/starts/indicatorsofukhousebuilding.xlsx", sheet_name="1b", skiprows=5)
ons["quarter"] = ons["Period"].apply(parse_quarter)
ons = ons.dropna(subset=["quarter"]).set_index("quarter").sort_index()
ons_fy = ons.index.map(fy_map)

components = ["New build completions", "Net conversions", "Net change of use",
             "Net other gains", "Demolitions", "Total net additional dwellings"]
lt120 = pd.read_excel("../../data/raw/net_additions/Live_Table_120.ods", sheet_name="LT120_unrounded", skiprows=4)
lt120 = lt120.set_index(lt120.columns[0]).T[components].iloc[:-2].apply(pd.to_numeric, errors="coerce")
avg = lt120.iloc[-4:-1].mean()

priv_actual = ons.groupby(ons_fy)["Completed - Private Enterprise"].sum().loc[2021:2023].mean()
non_private = avg["New build completions"] - priv_actual

net_add = lambda priv: (priv + non_private + avg["Net conversions"] + avg["Net change of use"]
                        - avg["Demolitions"] + avg["Net other gains"])

actual_back = ons.loc[pd.PeriodIndex(["2025Q2","2025Q3","2025Q4"], freq="Q"),
                      "Completed - Private Enterprise"].sum()


def run(path, log_col, strip_space=False):
    fc = pd.read_csv(path)
    if strip_space:
        fc["period"] = fc["period"].str.replace(" ", "", regex=False)
    if log_col == "ensemble_log" and "ensemble_log" not in fc:
        fc["ensemble_log"] = (fc["ardl_log"] + fc["nardl_log"]) / 2
    fc["ensemble_log"] = fc[log_col]
    
    ln_S_full = np.concatenate([seed, fc["ensemble_log"].values])
    quarters = pd.PeriodIndex(fc["period"], freq="Q")

    ln_C, lag_C = [], p["last_ln_C"]
    for t in range(len(fc)):
        lag_C = p["intercept"] + p["rho"]*lag_C + p["beta"]*ln_S_full[t] + seasonal[quarters[t].quarter]
        ln_C.append(lag_C)
    fc["completions"] = np.exp(ln_C) * p["smearing_factor"]
    fc["fy"] = quarters.map(fy_map)
    annual = fc.groupby("fy")["completions"].sum().rename("private_completions").to_frame().iloc[1:]
    annual["net_additions"] = net_add(annual["private_completions"])

    q1 = quarters[quarters.year == 2026][0]                    # robust to "2026 Q1" vs "2026Q1"
    priv_2025_26 = actual_back + fc.loc[quarters == q1, "completions"].iloc[0]

    return {"2024-25": 208600, "2025-26": net_add(priv_2025_26),
            "2026-27": annual.loc[2026, "net_additions"],
            "2027-28": annual.loc[2027, "net_additions"],
            "2028-29": annual.loc[2028, "net_additions"]}

rdl  = run("../../data/outputs/forecasts/obr_scenario_forecasts.csv", "ensemble_log")
vecm = run("../../data/outputs/forecasts/vecm_unconditional_forecast.csv", "vecm_log", strip_space=True)

target = 1_500_000
for name, d in [("ARDL/NARDL ensemble", rdl), ("VECM (unconditional)", vecm)]:
    print(name)
    for fy, v in d.items():
        print(f"{fy}: {v:,.0f}")
    cumulative = sum(d.values())
    print(f"Cumulative: {cumulative:,.0f} ({100*cumulative/target:.1f}% of {target:,}, "
          f"shortfall {target-cumulative:,.0f})\n")


non-private new build is held flat at its recent average (it won't respond to the policy scenarios), and conversions/change-of-use/demolitions are likewise held at 2021-24 averages. 

In [ ]:
import matplotlib.pyplot as plt

fy_start = lambda s: int(str(s)[:4])  # "2022-23" -> 2022

# Actual history from LT120, plus the 2024-25 actual already in delivery
actual = lt120["Total net additional dwellings"].dropna()
actual.index = actual.index.map(fy_start)
if 2024 not in actual.index:
    actual.loc[2024] = rdl["2024-25"]
actual = actual.sort_index()

# Forecast path (2025-26 onward)
def fcast_series(d):
    s = pd.Series({fy_start(k): v for k, v in d.items() if fy_start(k) >= 2025}).sort_index()
    return pd.concat([actual.iloc[[-1]], s])   # prepend last actual to close the gap

# Prepend the last actual point so the forecast line connects without a gap
join_rdl  = fcast_series(rdl)
join_vecm = fcast_series(vecm)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(actual.index, actual.values, color="#1f4e79", lw=2, label="Actual")
ax.plot(join_rdl.index,  join_rdl.values,  color="#c0392b", lw=2, ls="--",
        marker="o", label="ARDL/NARDL ensemble (OBR-conditioned)")
ax.plot(join_vecm.index, join_vecm.values, color="#e08e0b", lw=2, ls=":",
        marker="s", label="VECM (unconditional system)")
ax.set_ylabel("Net additional dwellings")
ax.set_xlabel("Financial year (start)")
ax.legend()
plt.tight_layout()
plt.show()

# Chronos

In [ ]:
fc_chr = pd.read_csv("../../data/outputs/forecasts/chronos_forward.csv")
fc_chr["log_starts"] = np.log(fc_chr["starts"])

ln_S_chr = np.concatenate([np.log(eng["starts"].dropna().iloc[-4:].values),
                            fc_chr["log_starts"].values])
quarters_chr = pd.PeriodIndex(fc_chr["Quarter"], freq="Q")

ln_C_chr, lag_C = [], p["last_ln_C"]   # reset seed -- don't reuse mutated lag_C
for t in range(len(fc_chr)):
    lag_C = p["intercept"] + p["rho"]*lag_C + p["beta"]*ln_S_chr[t] + seasonal[quarters_chr[t].quarter]
    ln_C_chr.append(lag_C)
fc_chr["completions"] = np.exp(ln_C_chr) * p["smearing_factor"]
fc_chr["fy"] = quarters_chr.map(fy_map)

annual_chr = fc_chr.groupby("fy")["completions"].sum().rename("private_completions").to_frame().iloc[1:]

annual_chr["net_additions"] = net_add(annual_chr["private_completions"])

priv_2025_26_chr = actual_back + fc_chr.loc[fc_chr["Quarter"] == "2026Q1", "completions"].iloc[0]

delivery_chr = {"2024-25": 208600, "2025-26": net_add(priv_2025_26_chr),
                 "2026-27": annual_chr.loc[2026, "net_additions"],
                 "2027-28": annual_chr.loc[2027, "net_additions"],
                 "2028-29": annual_chr.loc[2028, "net_additions"]}

cumulative_chr = sum(delivery_chr.values())
print("--- Chronos (unconditional) ---")
for fy, v in delivery_chr.items():
    print(f"{fy}: {v:,.0f}")
print(f"\nCumulative: {cumulative_chr:,.0f} ({100*cumulative_chr/target:.1f}% of {target:,})")

In [ ]:
import matplotlib.pyplot as plt
fc_chr.plot(x="Quarter", y="starts", marker="o", figsize=(8,4))
plt.title("Chronos forward starts, 2026Q1-2029Q1")
plt.show()
print(fc_chr[["Quarter","starts"]])